# Stage 3 Helmet Training (Google Colab)

Use a Colab GPU runtime. Before starting, upload the prepared `datasets/helmet/` folder (including `images/`, `labels/`, and `data.yaml`) to `MyDrive/traffic-ai/datasets/helmet/`. The source dataset and labels remain unchanged. This notebook copies the data to Colab's local disk for faster training, uses the same `yolo26n.pt`, 100 epochs, 640 image size, batch 8, workers 2, and patience 20, then saves the best model and run outputs back to Drive.


In [ ]:
!pip -q install ultralytics
from google.colab import drive
drive.mount('/content/drive')
import shutil, yaml
from pathlib import Path
drive_data = Path('/content/drive/MyDrive/traffic-ai/datasets/helmet')
local_data = Path('/content/helmet')
if not (drive_data / 'data.yaml').exists():
    raise FileNotFoundError(f'Upload datasets/helmet to {drive_data}')
if local_data.exists(): shutil.rmtree(local_data)
shutil.copytree(drive_data, local_data)
cfg = yaml.safe_load((local_data / 'data.yaml').read_text())
cfg.update(path=str(local_data), train='images/train', val='images/val', test='images/test')
(local_data / 'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print('GPU:', __import__('torch').cuda.get_device_name(0))
print('Dataset:', local_data)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n.pt')
train_results = model.train(
    data=str(local_data / 'data.yaml'), epochs=100, imgsz=640, batch=8, workers=2,
    patience=20, project='/content/runs', name='helmet_v1', device=0
)

In [ ]:
best = Path('/content/runs/helmet_v1/weights/best.pt')
assert best.exists(), 'Training did not produce best.pt'
out_dir = Path('/content/drive/MyDrive/traffic-ai/models/helmet')
out_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, out_dir / 'helmet_v1.pt')
shutil.copytree('/content/runs/helmet_v1', '/content/drive/MyDrive/traffic-ai/runs/helmet_v1', dirs_exist_ok=True)
print('Saved:', out_dir / 'helmet_v1.pt')

Download `helmet_v1.pt` from Drive and place it at `models/helmet/helmet_v1.pt` in the project, then run `python scripts/s3_evaluate_helmet.py`. Review the test results and validation previews before treating the model as useful.